In [1]:
import sys
from pyprojroot import here
sys.path.insert(0, str(here()))

In [ ]:
import numpy as np
import pandas as pd

from src.evaluation import (
    load_ground_truth,
    load_db_metadata,
    infer_bin_size_deg,
    filter_samples_with_masks,
    build_batch_queries,
    run_batch_coarse_scan,
    refine_query_with_dtw,
    summarize_results,
)

Set file paths and simple config values.

In [ ]:
db_path = here() / "notebooks" / "02_SkylineDatabase" / "output" / "skyline_db.parquet"
gt_path = here() / "data" / "synthetic_dataset" / "ground_truth.json"
masks_dir = here() / "data" / "synthetic_dataset" / "predicted_masks"

height_tolerance_m = 200.0
correct_dist_m = 500.0
compass_tolerance_deg = 20.0
weights = (0.33, 0.33, 0.33)
min_std_deg = 1.5
min_max_elev_deg = 1.0
dtw_window = 10
spatial_stride = 12
chunk_rows = 4000
limit = 20
use_altimeter = True
use_compass = True

Load ground truth and database metadata.

In [ ]:
gt_data, sample_ids = load_ground_truth(str(gt_path), limit=limit)
lon, lat, elev_m, n_vp = load_db_metadata(str(db_path))
bin_deg = infer_bin_size_deg(str(db_path))
valid_sids = filter_samples_with_masks(sample_ids, str(masks_dir))

print(f"Total candidate samples: {len(sample_ids)}")
print(f"Samples with predicted masks: {len(valid_sids)}")
print(f"Database viewpoints: {n_vp}")
print(f"Angular bin size: {bin_deg:.3f} deg")

Run matching and print the summary.

In [ ]:
print("Building query profiles...")
batch_queries = build_batch_queries(
    valid_sids,
    gt_data,
    str(masks_dir),
    bin_deg,
    n_vp,
    min_std_deg=min_std_deg,
    min_max_elev_deg=min_max_elev_deg,
    use_altimeter=use_altimeter,
    use_compass=use_compass,
)

print("Running coarse scan...")
run_batch_coarse_scan(
    batch_queries,
    str(db_path),
    elev_m,
    n_vp,
    chunk_rows=chunk_rows,
    spatial_stride=spatial_stride,
    weights=weights,
    compass_tolerance_deg=compass_tolerance_deg,
    height_tolerance_m=height_tolerance_m,
    progress_desc="Scanning DB"
)

print("Running fine refinement...")
all_results = []
for sample_id, query_state in batch_queries.items():
    result = refine_query_with_dtw(
        query_state,
        str(db_path),
        spatial_stride,
        n_vp,
        lat,
        lon,
        dtw_window=dtw_window,
        correct_dist_m=correct_dist_m,
    )
    if result is None:
        continue
    result["sample_id"] = sample_id
    all_results.append(result)

df_results = pd.DataFrame(all_results)
summary = summarize_results(df_results, len(valid_sids))

print("--- Baseline Evaluation Results ---")
print(f"Evaluated Queries: {summary['n_samples']}")
print(f"Skipped Queries (Flat/No Relief): {summary['skipped_flat']}")
print(f"Top-1 Accuracy (within 500m): {summary['top1_acc_500m']:.2f}%")
print(f"Top-5 Accuracy (within 500m): {summary['top5_acc_500m']:.2f}%")
print(f"Median Error: {summary['median_error_m']:.1f} meters")